# Model Optimization: Knowledge Distillation

In this notebook, we'll apply knowledge distillation techniques to our models using distributed processing. Instead of running the distillation on our notebook instance, we'll launch separate SageMaker Processing jobs to perform the distillation on more powerful instances.

## What is Knowledge Distillation?

Knowledge distillation is a model compression technique where a smaller "student" model is trained to mimic the behavior of a larger "teacher" model. The key insight is that the teacher's outputs contain rich information beyond just the hard labels - they contain the teacher's "dark knowledge" about the relationships between classes and the confidence in predictions.

### Benefits of Knowledge Distillation:
- **Smaller Models**: Student models have fewer parameters and are much smaller in size
- **Faster Inference**: Smaller models perform inference more quickly
- **Lower Resource Requirements**: Students require less memory and compute
- **Preserved Accuracy**: Students can retain much of the teacher's accuracy

### How Knowledge Distillation Works:
1. **Teacher Outputs**: The teacher model produces "soft targets" (probability distributions)
2. **Temperature Scaling**: These distributions are softened using a temperature parameter
3. **Student Training**: The student is trained to match both the correct labels and the teacher's soft targets
4. **Knowledge Transfer**: The student learns not just the correct answers but the teacher's reasoning

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform knowledge distillation on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
import os
import json
import torch
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput, Processor
from sagemaker.pytorch.processing import PyTorchProcessor

# Import our utility functions
from optimization_utils import analyze_job_failure, handle_processing_error, save_checkpoint

In [ ]:
,
    "import os\n",
    "import json\n",
    "import torch\n",
    "import time\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import boto3\n",
    "import sagemaker\n",
    "from sagemaker.processing import ProcessingInput, ProcessingOutput, Processor\n",
    "from sagemaker.pytorch.processing import PyTorchProcessor\n",
    "\n",
    "# Import our utility functions\n",
    "from optimization_utils import analyze_job_failure, handle_processing_error, save_checkpoint\n"

,
    "## 2. Load Workshop Settings\n",
    "\n",
    "Load the workshop settings that were configured in the first notebook."

In [ ]:
,
    "# Load stored variables\n",
    "%store -r S3_BUCKET\n",
    "%store -r AWS_REGION\n",
    "%store -r SAGEMAKER_ROLE_ARN\n",
    "%store -r OPTIMIZATION_INSTANCE_TYPE\n",
    "\n",
    "# Check if variables were successfully retrieved\n",
    "if 'S3_BUCKET' in locals() and S3_BUCKET != \"YOUR_BUCKET_NAME_HERE\":\n",
    "    print(\"Workshop settings loaded successfully:\")\n",
    "    print(f\"S3 Bucket: {S3_BUCKET}\")\n",
    "    print(f\"AWS Region: {AWS_REGION}\")\n",
    "    print(f\"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}\")\n",
    "    print(f\"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}\")\n",
    "else:\n",
    "    print(\"\u26a0\ufe0f Workshop settings not found or not configured.\")\n",
    "    print(\"Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.\")\n",
    "    \n",
    "    # Set default values that user should update\n",
    "    S3_BUCKET = \"YOUR_BUCKET_NAME_HERE\"  # Update this value\n",
    "    AWS_REGION = \"YOUR_REGION_HERE\"      # Update this value\n",
    "    SAGEMAKER_ROLE_ARN = \"YOUR_ROLE_ARN_HERE\"  # Update this value\n",
    "    OPTIMIZATION_INSTANCE_TYPE = \"ml.c5.xlarge\"  # Default optimization instance type\n",
    "    \n",
    "    # Store the updated values\n",
    "    %store S3_BUCKET\n",
    "    %store AWS_REGION\n",
    "    %store SAGEMAKER_ROLE_ARN\n",
    "    %store OPTIMIZATION_INSTANCE_TYPE\n"

,
    "## 3. Load Baseline Metrics and Model Information"

In [ ]:
,
    "# Load baseline metrics from file\n",
    "with open('baseline_metrics.json', 'r') as f:\n",
    "    baseline_metrics = json.load(f)\n",
    "\n",
    "print(f\"Loaded baseline metrics for {len(baseline_metrics)} models\")\n",
    "\n",
    "# Load model information from file\n",
    "with open('model_info.json', 'r') as f:\n",
    "    model_info = json.load(f)\n",
    "\n",
    "print(f\"Loaded information for {len(model_info)} models\")\n"

,
    "## 4. Define Sample Inputs for Each Task"

In [ ]:
,
    "# Define sample inputs for each task\n",
    "sample_inputs = {\n",
    "    \"sentiment_analysis\": \"I really enjoyed this movie. The acting was superb and the plot was engaging.\",\n",
    "    \"ner\": \"Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.\",\n",
    "    \"question_answering\": {\n",
    "        \"question\": \"What is machine learning?\",\n",
    "        \"context\": \"Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data.\"\n",
    "    },\n",
    "    \"masked_lm\": \"The [MASK

,
    "## 5. Examine and Upload Distillation Script to S3\n",
    "\n",
    "In this section, we'll examine and upload the Python script that performs the actual knowledge distillation. This script will be executed on the SageMaker Processing instances.\n",
    "\n",
    "### What the Script Does:\n",
    "1. **Loads the teacher model and tokenizer** from Hugging Face\n",
    "2. **Creates a smaller student model** with the same task capabilities\n",
    "3. **Prepares a dataset** for training the student model\n",
    "4. **Performs knowledge distillation** by training the student to mimic the teacher\n",
    "5. **Measures performance metrics** like model size and inference time\n",
    "6. **Saves the distilled model** and metrics to the output directory\n",
    "\n",
    "### Key Components of Knowledge Distillation:\n",
    "- **Temperature Parameter**: Controls how \"soft\" the teacher's probability distributions are\n",
    "- **KL Divergence Loss**: Measures how well the student matches the teacher's distributions\n",
    "- **Custom Trainer**: Implements the distillation loss function\n",
    "\n",
    "The script uses DistilBERT as the student model architecture, which is specifically designed for knowledge distillation from BERT-based models."

In [ ]:
,
    "# Display the distillation script with syntax highlighting\n",
    "%pycat distillation_script.py\n"

In [ ]:
,
    "# Upload the distillation script to S3\n",
    "s3_client = boto3.client('s3')\n",
    "s3_client.upload_file(\n",
    "    'distillation_script.py', \n",
    "    S3_BUCKET, \n",
    "    'scripts/distillation_script.py'\n",
    ")\n",
    "\n",
    "print(f\"Uploaded distillation script to s3://{S3_BUCKET}/scripts/distillation_script.py\")\n"

,
    "## 6. Launch Distributed Distillation Jobs\n",
    "\n",
    "Now we'll set up and launch the SageMaker Processing jobs to perform knowledge distillation. Each model will be processed in a separate job, allowing for parallel processing.\n",
    "\n",
    "### Distillation Process:\n",
    "1. **Create a PyTorch processor** with the appropriate instance type and configuration\n",
    "2. **For each model**:\n",
    "   - Save and upload model information to S3\n",
    "   - Define inputs (distillation script and model info) and outputs\n",
    "   - Launch a processing job with the appropriate arguments\n",
    "   - Store the job information for monitoring\n",
    "\n",
    "We're using DistilBERT as the student model architecture, which is about 40% smaller than BERT while retaining about 97% of its language understanding capabilities. The distillation process involves training for 3 epochs with a batch size of 8."

In [ ]:
,
    "# Define the instance type to use for distillation\n",
    "instance_type = OPTIMIZATION_INSTANCE_TYPE\n",
    "print(f\"Using instance type: {instance_type} for optimization jobs\")\n",
    "\n",
    "# Create a SageMaker session\n",
    "sagemaker_session = sagemaker.Session()\n",
    "\n",
    "# Create a PyTorch processor\n",
    "processor = PyTorchProcessor(\n",
    "    framework_version=\"1.13.1\",\n",
    "    py_version=\"py39\",\n",
    "    role=SAGEMAKER_ROLE_ARN,\n",
    "    instance_type=instance_type,\n",
    "    instance_count=1,\n",
    "    base_job_name=\"model-distillation\",\n",
    "    sagemaker_session=sagemaker_session,\n",
    "    # Add required packages\n",
    "    dependencies=[\"transformers\", \"datasets\"

In [ ]:
,
    "# Launch distillation jobs for each modeldistillation_jobs = {}for model_key in model_info.keys():\n",
    "    # Skip models that are not suitable for distillation    if model_info[model_key

,
    "## 7. Monitor Job Status\n",
    "\n",
    "After launching the distillation jobs, we need to monitor their progress. SageMaker Processing jobs run asynchronously, so we'll periodically check their status until all jobs are complete.\n",
    "\n",
    "### Monitoring Process:\n",
    "1. **Create a SageMaker client** to interact with the SageMaker API\n",
    "2. **Check job status every 30 seconds** until all jobs are complete\n",
    "3. **Display a status table** showing the current status of each job\n",
    "\n",
    "Knowledge distillation is more time-consuming than quantization or pruning because it involves training the student model. Depending on the model size and instance type, this process can take from several minutes to a few hours."

In [ ]:
,
    "# Monitor job status\n",
    "import import\n"

,
    "## 8. Collect Results\n",
    "\n",
    "Once all jobs are complete, we'll collect and combine the results from each job. Each job produces a metrics file containing information about the distilled model, such as size, inference time, and the comparison with the teacher model.\n",
    "\n",
    "### Collection Process:\n",
    "1. **Download metrics files** from S3 for each model\n",
    "2. **Combine metrics** into a single dictionary\n",
    "3. **Save combined metrics** to a local file for use in later notebooks\n",
    "\n",
    "This gives us a comprehensive view of the distillation results across all models, which we'll analyze in the next section."

In [ ]:
,
    "# Download and combine results\n",
    "distilled_metrics = {}\n",
    "\n",
    "for model_key in distillation_jobs.keys():\n",
    "    # Download metrics file\n",
    "    try:\n",
    "        s3_client.download_file(\n",
    "            S3_BUCKET,\n",
    "            f'optimization/outputs/{model_key}_distilled/distilled_metrics.json',\n",
    "            f'temp_{model_key}_distilled_metrics.json'\n",
    "        )\n",
    "        \n",
    "        # Load metrics\n",
    "        with open(f'temp_{model_key}_distilled_metrics.json', 'r') as f:\n",
    "            metrics = json.load(f)\n",
    "        \n",
    "        # Add to combined metrics\n",
    "        distilled_metrics.update(metrics)\n",
    "        \n",
    "        print(f\"Downloaded metrics for {model_key}\")\n",
    "    except Exception as e:\n",
    "        print(f\"Error downloading metrics for {model_key}: {e}\")\n",
    "\n",
    "    # Save combined metrics\n",
    "    with open('distilled_metrics.json', 'w') as f:\n",
    "    json.dump(distilled_metrics, f, indent=2)\n",
    "\n",
    "    print(f\"\\nSaved distilled metrics for {len(distilled_metrics)} models to distilled_metrics.json\")\n",
    "\n"

,
    "## 9. Compare Results\n",
    "\n",
    "Now we'll compare the performance of the distilled student models against the original teacher models. This comparison helps us understand the impact of knowledge distillation on model size and inference speed.\n",
    "\n",
    "### Key Metrics to Compare:\n",
    "- **Model Size**: How much smaller are the student models?\n",
    "- **Inference Time**: How much faster are the student models?\n",
    "- **Size Reduction Percentage**: The percentage reduction in model size\n",
    "- **Inference Speedup Percentage**: The percentage improvement in inference speed\n",
    "\n",
    "We expect to see significant size reductions (typically 40-60%) and inference speedups (typically 30-50%) with knowledge distillation. The exact improvements depend on the specific teacher and student architectures."

In [ ]:
,
    "# Create a DataFrame for comparison\n",
    "comparison_data = [

,
    "## 10. Deploy Models to SageMaker for Inference\n",
    "\n",
    "Now that we've created distilled student models, let's deploy them to SageMaker endpoints for real-world inference testing. We'll deploy both the original teacher models and the distilled student models to compare their performance.\n",
    "\n",
    "### Deployment Process:\n",
    "1. **Create model artifacts** in S3 for both teacher and student models\n",
    "2. **Create SageMaker models** using these artifacts\n",
    "3. **Create endpoints** to host the models\n",
    "4. **Wait for endpoint deployment** to complete\n",
    "\n",
    "This will allow us to directly compare inference performance between the teacher and student models."

In [ ]:
,
    "# Create a SageMaker client\n",
    "sagemaker_client = boto3.client('sagemaker')\n",
    "\n",
    "# Select a model to deploy for testing\n",
    "model_key = list(distilled_metrics.keys())[0

In [ ]:
,
    "# Wait for endpoints to be in service\n",
    "def wait_for_endpoint(endpoint_name):\n",
    "    status = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)['EndpointStatus'

,
    "## 11. Test Inference Performance\n",
    "\n",
    "Now that our endpoints are deployed, let's test their inference performance. We'll send the same input to both endpoints and measure:\n",
    "1. **Response time**: How long it takes to get a response\n",
    "2. **Throughput**: How many requests can be processed per second\n",
    "3. **Output quality**: Whether the outputs are similar between teacher and student models\n",
    "\n",
    "This will give us a real-world comparison of the performance benefits of knowledge distillation."

In [ ]:
,
    "# Create a SageMaker runtime client for inference\n",
    "runtime_client = boto3.client('sagemaker-runtime')\n",
    "\n",
    "# Prepare input data based on the model task\n",
    "task = distilled_metrics[model_key

,
    "## 12. Clean Up Endpoints\n",
    "\n",
    "To avoid unnecessary costs, let's clean up the endpoints we created. SageMaker endpoints incur charges as long as they're running, so it's important to delete them when they're no longer needed."

In [ ]:
,
    "# Delete endpoints and endpoint configurations\n",
    "print(\"Cleaning up endpoints...\")\n",
    "\n",
    "# Delete endpoints\n",
    "sagemaker_client.delete_endpoint(EndpointName=teacher_endpoint_name)\n",
    "sagemaker_client.delete_endpoint(EndpointName=student_endpoint_name)\n",
    "print(f\"Deleted endpoints: {teacher_endpoint_name}, {student_endpoint_name}\")\n",
    "\n",
    "# Delete endpoint configurations\n",
    "sagemaker_client.delete_endpoint_config(EndpointConfigName=teacher_endpoint_config_name)\n",
    "sagemaker_client.delete_endpoint_config(EndpointConfigName=student_endpoint_config_name)\n",
    "print(f\"Deleted endpoint configurations: {teacher_endpoint_config_name}, {student_endpoint_config_name}\")\n",
    "\n",
    "# Delete models\n",
    "sagemaker_client.delete_model(ModelName=teacher_model_name)\n",
    "sagemaker_client.delete_model(ModelName=student_model_name)\n",
    "print(f\"Deleted models: {teacher_model_name}, {student_model_name}\")\n",
    "\n",
    "print(\"\\nCleanup complete!\")\n"

,
    "## 13. Compare All Optimization Techniques\n",
    "\n",
    "Now that we've applied and tested all three optimization techniques (quantization, pruning, and knowledge distillation), let's compare their effectiveness.\n",
    "\n",
    "### Comparison Metrics:\n",
    "- **Model Size Reduction**: How much smaller is the optimized model?\n",
    "- **Inference Speed Improvement**: How much faster is the optimized model?\n",
    "- **Output Quality**: How similar are the outputs to the original model?\n",
    "- **Implementation Complexity**: How difficult is it to implement the technique?\n",
    "- **Training Requirements**: Does the technique require additional training?\n",
    "\n",
    "This comparison will help us understand which optimization technique is best suited for different use cases."

In [ ]:
,
    "# Load metrics from all optimization techniques\n",
    "try:\n",
    "    with open('quantized_metrics.json', 'r') as f:\n",
    "        quantized_metrics_all = json.load(f)\n",
    "    print(f\"Loaded quantized metrics for {len(quantized_metrics_all)} models\")\n",
    "except FileNotFoundError:\n",
    "    quantized_metrics_all = {}\n",
    "    print(\"No quantized metrics found\")\n",
    "\n",
    "try:\n",
    "    with open('pruned_metrics.json', 'r') as f:\n",
    "        pruned_metrics_all = json.load(f)\n",
    "    print(f\"Loaded pruned metrics for {len(pruned_metrics_all)} models\")\n",
    "except FileNotFoundError:\n",
    "    pruned_metrics_all = {}\n",
    "    print(\"No pruned metrics found\")\n",
    "\n",
    "# We already have distilled_metrics loaded\n",
    "print(f\"Loaded distilled metrics for {len(distilled_metrics)} models\")\n",
    "\n",
    "# Create a comparison table\n",
    "comparison_data = [

,
    "## 14. Visualization of Optimization Results"

In [ ]:
,
    "# Create visualizations to compare optimization techniques\n",
    "plt.figure(figsize=(12, 6))\n",
    "\n",
    "# Group by technique and calculate mean values\n",
    "technique_means = comparison_df.groupby('Technique').mean()\n",
    "\n",
    "# Size reduction plot\n",
    "plt.subplot(1, 2, 1)\n",
    "sns.barplot(x=technique_means.index, y='Size Reduction (%)', data=technique_means)\n",
    "plt.title('Average Model Size Reduction')\n",
    "plt.ylabel('Size Reduction (%)')\n",
    "plt.ylim(0, 100)\n",
    "\n",
    "# Speed improvement plot\n",
    "plt.subplot(1, 2, 2)\n",
    "sns.barplot(x=technique_means.index, y='Speed Improvement (%)', data=technique_means)\n",
    "plt.title('Average Inference Speed Improvement')\n",
    "plt.ylabel('Speed Improvement (%)')\n",
    "plt.ylim(0, 100)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n"

,
    "## 15. Conclusion and Recommendations\n",
    "\n",
    "Based on our experiments with quantization, pruning, and knowledge distillation, we can make the following recommendations:\n",
    "\n",
    "### Quantization\n",
    "- **Best for**: Quick optimization with minimal effort\n",
    "- **Advantages**: No training required, easy to implement\n",
    "- **Disadvantages**: Limited size reduction, potential accuracy impact\n",
    "- **Recommended when**: You need a quick solution with minimal development effort\n",
    "\n",
    "### Pruning\n",
    "- **Best for**: Moderate optimization with some development effort\n",
    "- **Advantages**: Good size reduction, minimal accuracy impact\n",
    "- **Disadvantages**: Requires careful tuning of pruning parameters\n",
    "- **Recommended when**: You need a balance between optimization and development effort\n",
    "\n",
    "### Knowledge Distillation\n",
    "- **Best for**: Maximum optimization with significant development effort\n",
    "- **Advantages**: Greatest size reduction and speed improvement\n",
    "- **Disadvantages**: Requires training, most complex to implement\n",
    "- **Recommended when**: You need the most optimized model and can invest in training\n",
    "\n",
    "### Combined Approach\n",
    "For the best results, consider combining these techniques:\n",
    "1. Start with knowledge distillation to create a smaller student model\n",
    "2. Apply pruning to remove unnecessary weights\n",
    "3. Finally, apply quantization to reduce precision\n",
    "\n",
    "This combined approach can yield the greatest optimization benefits while maintaining acceptable accuracy."

,
    "## 16. Resource Cleanup\n",
    "\n",
    "To avoid unnecessary costs, make sure all SageMaker resources have been cleaned up:\n",
    "- Endpoints\n",
    "- Endpoint configurations\n",
    "- Models\n",
    "- Processing jobs\n",
    "\n",
    "You can use the AWS Management Console or the AWS CLI to verify that all resources have been properly deleted."